#### Chroma DB

Chroma is a AI-native open-source database focused on developer productivity and happiness.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [4]:
## building a sample vectordb

from langchain_chroma import Chroma

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
loader = TextLoader("sample.txt", encoding="utf-8")

data = loader.load()

data

[Document(metadata={'source': 'sample.txt'}, page_content='The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. \nOur model achieves 28.4 BLEU on the WMT 2014 Englishto-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.0 after training for 3.5 days on eight GPUs, a small fraction of the training costs of t

In [8]:
# Splitting the data into chunks

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)

splits = text_splitter.split_documents(data)

In [9]:
## Create vector db with embeddings

embeddings = HuggingFaceEmbeddings( model_name="all-MiniLM-L6-v2")

vectordb = Chroma.from_documents(splits, embedding=embeddings)

vectordb

C:\Users\teler\AppData\Local\Temp\ipykernel_15280\2104142179.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings( model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1332.32it/s]


In [10]:
## querying the vector db

query = "What major architectural components, commonly used in existing state-of-the-art models, does the proposed Transformer completely eliminate?"

docs = vectordb.similarity_search(query)

docs[0].page_content

'In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'

In [11]:
## Saving to the local disk
vectordb = Chroma.from_documents(splits, embedding=embeddings, persist_directory="./chroma_index")

In [15]:
## loading the vector db from the local disk

db2 = Chroma(persist_directory="./chroma_index", embedding_function=embeddings)

docs = db2.similarity_search(query)

docs[0].page_content

'In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'

In [17]:
# Similarity Search With Score
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='a488d48e-ba06-4c20-af44-d483081f7c2d', metadata={'source': 'sample.txt'}, page_content='In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'),
  1.1526060104370117),
 (Document(id='ac252298-1ed6-4985-947d-fa51f0815772', metadata={'source': 'sample.txt'}, page_content='more parallelizable and requiring significantly less time to train.'),
  1.5245757102966309),
 (Document(id='3fe87692-50c5-4d0d-ab3b-e5bb6248684e', metadata={'source': 'sample.txt'}, page_content='The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect th

In [ ]:
## Retriever Options

retriever = vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'